# Interior model: run a job on a Colab GPU
1. Runtime -> Change runtime type -> pick a GPU (A100 or L4 for training, T4 for small checks).
2. Secrets (key icon, left): `HF_TOKEN` with write access to your private dataset repos, notebook access on.
3. Set the job in the next cell, then Runtime -> Run all. Keep this tab open (on the Pro plan a closed tab stops the run).
Results are uploaded to the private Hugging Face repo by the job itself, as on Hugging Face Jobs.

In [ ]:
# ---- the job (same names and settings as research/interior_model/jobs/launch_script.py)
JOB = 'elf_gen_job.py'              # a script in research/interior_model/jobs/
REPO_SUFFIX = 'hanseg-ct-mr'         # your private dataset repo (the job's first argument)
COMMIT = 'main'                      # code version: 'main' or a commit hash
ENV = {                              # the job's settings (env), as on Hugging Face Jobs
    'TAG': 'colab_trial',
    'GENS': 'NL30k=field_exp/suite3/NL_s0/flow.pt',
    'AE': 'field_exp/ae_check2/tvae_s1/ae_soft.pt',
}

In [ ]:
# ---- setup: GPU, packages, token (from Colab Secrets only), code version
import os, subprocess, sys, json, time, urllib.request
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'NO GPU: set Runtime -> Change runtime type')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=1.0', 'hf_xet', 'scikit-image', 'scipy', 'SimpleITK', 'nibabel'], check=True)
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import HfApi
user = HfApi().whoami()['name']; repo = '%s/%s' % (user, REPO_SUFFIX)
print('private repo', repo, 'private:', HfApi().repo_info(repo, repo_type='dataset').private)
if COMMIT == 'main':
    COMMIT = json.load(urllib.request.urlopen('https://api.github.com/repos/gardianofthedarkness/model-to-vrm/commits/main'))['sha']
print('code version', COMMIT)

In [ ]:
# ---- run the job (output streams below; the job uploads its results itself)
work = '/content/run'; os.makedirs(work, exist_ok=True)
src = 'https://raw.githubusercontent.com/gardianofthedarkness/model-to-vrm/%s/research/interior_model/jobs/%s' % (COMMIT, JOB)
urllib.request.urlretrieve(src, os.path.join(work, JOB))
env = dict(os.environ, COMMIT=COMMIT, PYTHONUNBUFFERED='1', **ENV)
t0 = time.time()
p = subprocess.Popen([sys.executable, '-u', JOB, repo], cwd=work, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'it/s]' not in line and 'B/s]' not in line: print(line.rstrip()[:1500], flush=True)
p.wait(); print('job exit', p.returncode, 'after %.1f min' % ((time.time() - t0) / 60))

In [ ]:
# ---- what the job saved in the private repo
tag = ENV.get('TAG') or ENV.get('SUITE') or ''
files = [f for f in HfApi().list_repo_files(repo, repo_type='dataset') if tag and ('/' + tag + '/') in ('/' + f)]
print(len(files), 'files'); print('\n'.join(files[:40]))